# CeNN P-Delta2 — Beat-the-Transformer Colab

Tests a curvature-aware **Preconditioned Delta2** layer against frozen SmolLM2 attention. The balanced profile pre-registers seven scenarios: current Delta2, pure P-Delta2 at 64/96/128 features, P-Delta2 + exact local windows, and a Delta2-window control. Selection is validation-only; test scoring is locked afterward.

Success is either a **strict held-out NLL win** (paired 95% bootstrap interval below zero) or **memory-efficient parity** (inside ±0.02 nats with <50% of FP16 Transformer KV state). It also tests 256/512/1024-token contexts.

Research basis: [Preconditioned DeltaNet](https://arxiv.org/abs/2604.21100), [Gated DeltaNet-2](https://arxiv.org/abs/2605.22791), and [Kimi Linear](https://arxiv.org/abs/2510.26692). This is an independent research adaptation, not a reproduction of fused paper kernels.


In [ ]:
import pathlib, subprocess, sys
REF='codex/pdelta2-beat-transformer-20260914'
REPO=pathlib.Path('/content/TinyCeNN-LM')
if REPO.exists():
    subprocess.run(['git','-C',str(REPO),'fetch','origin'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','-B','pdelta2-run',f'origin/{REF}'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'transformers==4.57.6','datasets>=3,<5','pandas','matplotlib'],check=True)
print('Commit:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True).strip())


In [ ]:
PROFILE='balanced'  # quick | balanced | strong
LAYER=18
TRAIN_CONTEXT=256
TEST_CONTEXTS='256,512,1024'
OUTDIR=REPO/'result'/'pdelta2-beat-transformer-colab'
cmd=[sys.executable,'-m','scripts.benchmark_pdelta2_beat_transformer','--profile',PROFILE,'--layer',str(LAYER),'--train-context',str(TRAIN_CONTEXT),'--test-contexts',TEST_CONTEXTS,'--output-dir',str(OUTDIR)]
print(' '.join(cmd))
subprocess.run(cmd,check=True,cwd=REPO)


In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
val=pd.read_csv(OUTDIR/'validation_summary.csv')
test=pd.read_csv(OUTDIR/'test_summary.csv')
report=json.loads((OUTDIR/'pdelta2_report.json').read_text())
print('Validation-selected winner:',report['winner_selected_on_validation'])
display(val.round(6)); display(test.round(6))
r=test[test.context==TRAIN_CONTEXT]
fig,ax=plt.subplots(figsize=(8,5)); ax.scatter(r.state_vs_transformer_fp16,r.delta_nll,s=80)
for _,x in r.iterrows(): ax.annotate(x.candidate,(x.state_vs_transformer_fp16,x.delta_nll),xytext=(4,4),textcoords='offset points',fontsize=8)
ax.axhline(0,linewidth=1); ax.axhline(.02,linewidth=1,linestyle='--'); ax.axhline(-.02,linewidth=1,linestyle='--')
ax.set_xlabel('Persistent state / Transformer FP16 KV cache'); ax.set_ylabel('ΔNLL vs frozen Transformer'); ax.set_title('P-Delta2 quality–memory Pareto screen'); plt.show()


## Interpretation

`strict_quality_win` is the strongest signal. `memory_efficient_parity` is also important because P-Delta2 state is bounded while Transformer KV state grows with context. If this wins, the next rigorous step is multi-layer replacement plus a matched Transformer adaptation control. Speed is not the target of this reference PyTorch implementation; fused Triton/CUDA work should follow only after quality succeeds.
